# Chapter 7: Planning and Reflection


In [ ]:
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2


## 7.1 Task Data Structure


In [ ]:
from scratch_agent.planning import Task

# Create tasks with different statuses
tasks = [
    Task(content="Research the history of artificial intelligence", status="completed"),
    Task(content="Summarize key milestones", status="in_progress"),
    Task(content="Write a brief report", status="pending"),
]

for t in tasks:
    print(str(t))

## 7.2 Planning Tool


In [ ]:
from scratch_agent.planning import create_tasks, Task
from scratch_agent.context import ExecutionContext

# Inspect the create_tasks tool
print(f"Tool name: {create_tasks.name}")
print(f"Description: {create_tasks.description[:100]}...")

# Demo: execute create_tasks
ctx = ExecutionContext()
result = await create_tasks(ctx, tasks=[
    {"content": "Search for information about quantum computing", "status": "in_progress"},
    {"content": "Summarize the key findings", "status": "pending"},
    {"content": "Write a brief report", "status": "pending"},
])
print(f"\nPlan:\n{result}")

## 7.3 Agent with Planning


In [ ]:
from scratch_agent.agent import Agent
from scratch_agent.llm import LlmClient
from scratch_agent.planning import create_tasks
from scratch_agent.tools.base import tool

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

# Create an agent with planning capability
planning_agent = Agent(
    model=LlmClient(model="gpt-4o-mini"),
    tools=[create_tasks, calculator],
    instructions="You are a research assistant. Break complex tasks into steps using create_tasks, then execute each step.",
    max_steps=8,
)

result = await planning_agent.run(
    "Calculate the sum of the first 10 prime numbers, then multiply by 3."
)
print(f"Result: {result.output}")

## 7.4 Reflection Tool


In [ ]:
from scratch_agent.planning import reflection
from scratch_agent.context import ExecutionContext

# Inspect the reflection tool
print(f"Tool name: {reflection.name}")
print(f"Description: {reflection.description[:100]}...")

# Demo: execute reflection
ctx = ExecutionContext()
result1 = await reflection(ctx, analysis="Found two conflicting data sources. Need to verify which is correct.")
print(f"\nNormal reflection: {result1}")

result2 = await reflection(ctx, analysis="Search tool returned no results. Need alternative approach.", need_replan=True)
print(f"Replan reflection: {result2}")

## 7.5 Planning + Reflection Combined

In [ ]:
from scratch_agent.agent import Agent
from scratch_agent.llm import LlmClient
from scratch_agent.planning import create_tasks, reflection
from scratch_agent.tools.base import tool

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

# Agent with both planning and reflection
full_agent = Agent(
    model=LlmClient(model="gpt-4o-mini"),
    tools=[create_tasks, reflection, calculator],
    instructions="""You are a research assistant with planning and reflection capabilities.
1. Use create_tasks to break down complex problems into steps.
2. Execute each step using available tools.
3. Use reflection to evaluate your progress and adjust your approach.""",
    max_steps=10,
)

result = await full_agent.run(
    "What is the sum of squares of 7, 11, and 13? Verify your answer by computing each square individually."
)
print(f"Result: {result.output}")